# Évaluation des Prévisions de Croissance du FMI

Ce notebook analyse l'écart entre les prévisions de croissance publiées historiquement par le FMI dans ses éditions d'avril et d'octobre du WEO et la croissance réellement observée par la suite.

In [ ]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

plt.style.use('seaborn-v0_8-whitegrid')

# Jupyter lance le noyau dans notebooks/ : la racine du dépôt est le premier dossier,
# en remontant, qui contient le paquet pib. Les données sont lues depuis là.
RACINE = next(d for d in [Path.cwd(), *Path.cwd().parents] if (d / 'pib').is_dir())
PROCESSED_DIR = RACINE / 'data' / 'processed'
EVAL_FILE = PROCESSED_DIR / 'weo_forecast_evaluation_ngdp_rpch.csv'
BIAS_FILE = PROCESSED_DIR / 'weo_forecast_bias_ngdp_rpch.csv'

In [ ]:
# Vérification de l'existence des fichiers
if not os.path.exists(EVAL_FILE) or not os.path.exists(BIAS_FILE):
    raise FileNotFoundError("Fichiers d'évaluation introuvables. Veuillez d'abord exécuter `python -m pib.evaluate_forecasts` depuis la racine.")

# Chargement des données
df_eval = pd.read_csv(EVAL_FILE)
df_bias = pd.read_csv(BIAS_FILE)

print(f"Données chargées : {len(df_eval):,} projections évaluées.")

## 1. Biais et Erreur Moyenne selon l'Horizon de Prévision

Plus on s'éloigne de l'année visée, plus la prévision est incertaine. L'horizon `1` représente l'année suivante, `5` représente une prévision à 5 ans.

In [ ]:
display(df_bias)

# Visualisation de l'erreur absolue moyenne
plt.figure(figsize=(10, 6), dpi=100)
sns.barplot(data=df_bias, x='horizon', y='erreur_absolue_moyenne', hue='reference')
plt.title('Erreur absolue moyenne par horizon de prévision')
plt.ylabel('Points de croissance (%)')
plt.xlabel('Horizon (années)')
plt.tight_layout()
plt.show()

## 2. Distribution des Erreurs (Prévision à 1 an)

Comment se répartissent les erreurs pour l'année suivante (Horizon = 1) ?

In [ ]:
df_h1 = df_eval[df_eval['horizon'] == 1].dropna(subset=['erreur_vs_fmi']).copy()

plt.figure(figsize=(10, 6), dpi=100)
sns.histplot(df_h1['erreur_vs_fmi'], bins=50, kde=True, color='indigo')
plt.axvline(0, color='black', linestyle='--')
plt.title('Distribution des erreurs de prévision à 1 an (vs. Ré-estimation FMI)')
plt.xlabel('Erreur (Prévision - Réalisé, en points de %)\n> 0 = Trop optimiste, < 0 = Trop pessimiste')
plt.ylabel('Nombre de projections')
plt.tight_layout()
plt.show()

print("Statistiques descriptives de l'erreur à 1 an : ")
display(df_h1['erreur_vs_fmi'].describe())

## 3. Les Plus Gros Ratés (Erreurs Supérieures à 10 points)

Quelles ont été les pires surprises économiques pour le FMI à l'horizon d'un an ?

In [ ]:
rates = df_h1[df_h1['erreur_vs_fmi'].abs() > 10].copy()
rates = rates.sort_values('erreur_vs_fmi', key=abs, ascending=False)

rates['Surprise'] = np.where(rates['erreur_vs_fmi'] > 0, 'Déception (Croissance < Prévue)', 'Bonne surprise (Croissance > Prévue)')

display(rates[['country', 'year', 'vintage', 'valeur', 'realise_fmi', 'erreur_vs_fmi', 'Surprise']].head(15))